In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [3]:
import json
from birddog.database import Database
from birddog.runtime import Runtime
from birddog.database_updater import DatabaseUpdater
from birddog.wiki import (
    get_root_label,
    page_label,
    sequential_page_label,
    _read_wiki_text,
    )

2026-07-10 09:01:26,224 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-07-10 09:01:26,231 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-07-10 09:01:26,340 [INFO] Translation is enabled. Using GCP translator
2026-07-10 09:01:26,341 [INFO] Using Google Cloud translation API
2026-07-10 09:01:26,342 [INFO] GoogleCloudTranslator using REST API


In [15]:
db = Database()

2026-07-10 09:08:50,256 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-07-10 09:08:50,440 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   22.00     0.03    39.00       0.00           24


In [10]:
fname = "../ollama/wiki_urls_flat.json"
with open(fname) as file:
    url_list = json.loads(file.read())

In [11]:
url_list

['https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6709_Список_євреїв_м-ка_Ямпіль,_які_підлягають_переселенню_з_території_50-вірстної_зони_(1846).pdf',
 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6712_Списки_євреїв_м-ка_Ожегівці,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf',
 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6711_Списки_євреїв_м-ка_Кульчини,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf',
 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_491-94-55._1854._Інвентар_казеного_Трипільського_маєтку_Київської_губернії.pdf',
 'https://uk.wikisource.org/wiki/Файл:DAKO_491_96_133.pdf',
 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_491-96-263._1847-1858._Люстрація_Трипільського_казенного_маєтку_Київського_повіту.pdf',
 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_515-1-104._1871-1873._Люстрації_сіл_Севастянівка_та_Івангород_Гайсинського_повіту_Подільської_губернії.pdf',
 'https://uk.wikisource.org/wiki/File:ЦДІАК_28-1-

In [12]:
rec_ids = db.lookup("ML Document Set", url_list)

2026-07-10 09:06:57,301 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   21.00     0.01    39.00       0.00           24


In [13]:
rec_ids

[7830,
 8246,
 8384,
 8418,
 8522,
 7809,
 7184,
 7166,
 7412,
 7127,
 7105,
 7305,
 7420,
 7101,
 449]

In [16]:
recs = db.read("ML Document Set", rec_ids, fields=["url", "title"])

In [17]:
recs

[{'Id': 7830,
  'title': 'Файл:ЦДІАК_442-1-6709_Список_євреїв_м-ка_Ямпіль,_які_підлягають_переселенню_з_території_50-вірстної_зони_(1846).pdf',
  'url': 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6709_Список_євреїв_м-ка_Ямпіль,_які_підлягають_переселенню_з_території_50-вірстної_зони_(1846).pdf'},
 {'Id': 8246,
  'title': 'Файл:ЦДІАК_442-1-6712_Списки_євреїв_м-ка_Ожегівці,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf',
  'url': 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6712_Списки_євреїв_м-ка_Ожегівці,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf'},
 {'Id': 8384,
  'title': 'Файл:ЦДІАК_442-1-6711_Списки_євреїв_м-ка_Кульчини,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf',
  'url': 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6711_Списки_євреїв_м-ка_Кульчини,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf'},
 {'Id': 8418,
  'title': 'Файл:Ц

In [21]:
prefix = "https://uk.wikisource.org/wiki/"
for rec in recs:
    url = rec.get("url") or ""
    if url.startswith(prefix):
        try:
            title = url[len(prefix):]
            print(f"fetching {url}")
            wiki_text = _read_wiki_text(title)
            if wiki_text:
                rec["wiki_text"] = wiki_text
        except:
            print(f"......failed on {url}")

fetching https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6709_Список_євреїв_м-ка_Ямпіль,_які_підлягають_переселенню_з_території_50-вірстної_зони_(1846).pdf
......failed on https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6709_Список_євреїв_м-ка_Ямпіль,_які_підлягають_переселенню_з_території_50-вірстної_зони_(1846).pdf
fetching https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6712_Списки_євреїв_м-ка_Ожегівці,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf
......failed on https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6712_Списки_євреїв_м-ка_Ожегівці,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf
fetching https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6711_Списки_євреїв_м-ка_Кульчини,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf
......failed on https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6711_Списки_євреїв_м-ка_Кульчини,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зон

In [22]:
recs

[{'Id': 7830,
  'title': 'Файл:ЦДІАК_442-1-6709_Список_євреїв_м-ка_Ямпіль,_які_підлягають_переселенню_з_території_50-вірстної_зони_(1846).pdf',
  'url': 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6709_Список_євреїв_м-ка_Ямпіль,_які_підлягають_переселенню_з_території_50-вірстної_зони_(1846).pdf'},
 {'Id': 8246,
  'title': 'Файл:ЦДІАК_442-1-6712_Списки_євреїв_м-ка_Ожегівці,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf',
  'url': 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6712_Списки_євреїв_м-ка_Ожегівці,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf'},
 {'Id': 8384,
  'title': 'Файл:ЦДІАК_442-1-6711_Списки_євреїв_м-ка_Кульчини,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf',
  'url': 'https://uk.wikisource.org/wiki/Файл:ЦДІАК_442-1-6711_Списки_євреїв_м-ка_Кульчини,_що_підлягають_переселенню_з_території_50-вірстної_прикордонної_зони_(1846).pdf'},
 {'Id': 8418,
  'title': 'Файл:Ц